# 지도학습 분류와 회귀

## 한글 폰트 설정 (가장 먼저 한 번 실행)

matplotlib의 기본 폰트(`DejaVu Sans`)에는 한글 글리프가 없어, plot의 한글 제목/축/범례가 박스(□)로 깨지고 `UserWarning: Glyph ... missing from font(s) DejaVu Sans` 가 뜹니다. 아래 셀이 시스템에 설치된 한글 폰트를 자동으로 감지해 `matplotlib.rcParams['font.family']` 에 설정합니다 — macOS는 `AppleGothic`, Windows는 `Malgun Gothic`, Linux/Colab은 `NanumGothic` 또는 `Noto Sans CJK KR` 가 우선 시도됩니다. 한글 폰트가 시스템에 없을 때는 친절한 설치 안내가 출력되니 그대로 따라 한 뒤 커널을 재시작하세요.

In [ ]:
import matplotlib
import matplotlib.font_manager as fm

_korean_fonts = [
    "AppleGothic",          # macOS
    "Apple SD Gothic Neo",  # macOS (newer)
    "Malgun Gothic",        # Windows
    "NanumGothic",          # Linux / Colab (apt: fonts-nanum)
    "Nanum Gothic",
    "Noto Sans CJK KR",     # Linux (Noto family)
    "Noto Sans KR",
    "UnDotum",
]
_available = {f.name for f in fm.fontManager.ttflist}
_picked = next((f for f in _korean_fonts if f in _available), None)
if _picked:
    matplotlib.rcParams["font.family"] = _picked
    print(f"matplotlib 한글 폰트 = {_picked}")
else:
    print("⚠ 한글 폰트를 찾지 못했습니다. plot 라벨이 깨질 수 있습니다.")
    print("  Linux/Colab: !apt-get install -y fonts-nanum && fc-cache -fv  → 커널 재시작")
    print("  macOS:       시스템 기본 AppleGothic 이 자동 인식돼야 합니다 (matplotlib 캐시 갱신 필요시 ~/.matplotlib/fontList.json 삭제)")
    print("  Windows:     Malgun Gothic 이 OS 기본 폰트로 잡힙니다")
matplotlib.rcParams["axes.unicode_minus"] = False  # 마이너스 부호 깨짐 방지


# Step 1 — load_digits로 데이터 진입과 4줄 점검 + 샘플 이미지 시각화

**목표**: digits 데이터셋을 로드하고 shape · dtype · 클래스 분포를 확인한 뒤 첫 10장의 손글씨를 시각화해 *어떤 문제인지* 눈으로 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
print("=== digits 점검 ===")
print("X.shape =", X.shape)                      # (1797, 64)
print("X.dtype =", X.dtype)                      # float64
print("y.shape =", y.shape)                      # (1797,)
print("y unique =", np.unique(y))                # [0 1 2 3 4 5 6 7 8 9]
print("y counts =", np.bincount(y))              # 클래스별 sample 수
print("X range =", X.min(), "~", X.max())        # 0.0 ~ 16.0 (32×32 bitmap을 4×4 블록 단위로 나누고, 각 블록 안의 켜진 픽셀 수를 세어서 8×8 데이터로 만든 것이기 때문)

In [ ]:
# 시각화 — 첫 10장의 손글씨를 8×8 grayscale로
fig, axes = plt.subplots(2, 5, figsize=(8, 3.5))
for ax, img, label in zip(axes.ravel(), digits.images[:10], y[:10]):
    ax.imshow(img, cmap="gray_r"); ax.set_title(f"y={label}")
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout(); plt.show()

# Step 2 — train/test split

**목표**: 1797장을 train · test 로 절반씩 나누고 각 split의 shape을 확인합니다.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, shuffle=False
)
# np.bincount()는 0 이상의 정수 값들이 각각 몇 번 나왔는지 세는 함수
print("X_train.shape =", X_train.shape, "  y_train counts =", np.bincount(y_train))
print("X_test.shape  =", X_test.shape,  "  y_test  counts =", np.bincount(y_test))

# Step 3 — 5개 분류기 일괄 fit / score 정확도 표

**목표**: 5개 estimator 가족(k-NN / Tree / LogReg / SVC / RF)을 같은 train / test 데이터에 한 번에 fit / score 해 정확도 비교 표를 만듭니다. scaling 정책(거리 기반 → Pipeline, 분기 기반 → 단독)을 코드로 표현합니다.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
estimators = {
    "k-NN (k=5)":      make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Tree (depth=10)": DecisionTreeClassifier(max_depth=10, random_state=0),
    "LogisticReg":     make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "SVC (rbf)":       make_pipeline(StandardScaler(), SVC(kernel="rbf")),
    "RandomForest":    RandomForestClassifier(n_estimators=100, random_state=0),
}

In [ ]:
results = {}
for name, est in estimators.items():
    est.fit(X_train, y_train)
    results[name] = est.score(X_test, y_test)
    print(f"  {name:>18s}: acc = {results[name]:.4f}")


# Step 4 — best 분류기의 classification_report + ConfusionMatrixDisplay

**목표**: 정확도 최고 모델로 `classification_report` 와 `ConfusionMatrixDisplay` 를 띄워 클래스별 precision / recall / f1과 오분류가 몰리는 칸을 봅니다 (개념 §4.1 평가 도구).

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [ ]:
best_name = max(results, key=results.get)
best_est  = estimators[best_name]
y_pred    = best_est.predict(X_test)
print(f"=== {best_name} ===")
print(classification_report(y_test, y_pred))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.show()

# 분류기별 상세 실행

## KNeighborsClassifier — n_neighbors 효과

**목표**: 같은 데이터에 `n_neighbors=1` 과 `n_neighbors=15` 두 가지로 fit해 boundary smoothness 차이가 정확도에 어떻게 반영되는지 봅니다.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_pipe = lambda k: make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))

for k in [1, 3, 5, 15, 50]:
    acc = knn_pipe(k).fit(X_train, y_train).score(X_test, y_test)
    print(f"  k={k:3d}: acc = {acc:.4f}")


**해석**: k=1은 가장 가까운 한 이웃만 보므로 noise까지 따라 약간 흔들리고, k가 너무 크면(50) 너무 멀리 있는 이웃까지 평균에 들어와 정확도가 떨어집니다. 일반적으로 k=3~10이 baseline이며 cv로 best k를 선택합니다.

## DecisionTreeClassifier + export_text — 분기 규칙 시각화

**목표**: 트리를 iris(작은 데이터)에 fit하고 `export_text` 로 학습된 분기 규칙을 ASCII로 직접 봅니다.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier, export_text

In [ ]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
clf_tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_iris, y_iris)
print(f"train acc = {clf_tree.score(X_iris, y_iris):.4f}")
print(export_text(clf_tree, feature_names=list(iris.feature_names)))

**해석 + 다음 단계**: `petal length <= 2.45` 한 줄로 setosa(class 0)가 분리됩니다 — iris의 첫 분기 규칙이 *꽃잎 길이 2.45 cm 이상인가* 라는 단순 규칙임을 직접 확인했습니다. 트리는 학습 결과가 *읽을 수 있는 규칙* 으로 저장 됩니다.

## LogisticRegression — predict_proba + coef_

**목표**: digits에 LogReg를 fit해 클래스별 확률(`predict_proba`)과 학습 가중치(`coef_`)의 shape을 직접 확인합니다.

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
clf_lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
acc = clf_lr.score(X_test, y_test)
print(f"LogisticRegression acc = {acc:.4f}")

In [ ]:
# Pipeline에서 LogReg 단계만 꺼내 coef_ 보기
inner_lr = clf_lr.named_steps["logisticregression"]
print("coef_.shape       =", inner_lr.coef_.shape)        # (10, 64)
print("intercept_.shape  =", inner_lr.intercept_.shape)   # (10,)

In [ ]:
# 첫 5개 test 샘플의 클래스별 확률
proba = clf_lr.predict_proba(X_test[:5])
print("predict_proba.shape =", proba.shape)                # (5, 10)
print("첫 샘플의 top-3 확률 =", np.sort(proba[0])[-3:].round(3))

**해석**: `coef_` shape `(10, 64)` 는 클래스 10개 × feature 64개의 학습 가중치입니다 — k-NN / SVC에는 없는 *해석 가능성* 입니다. `predict_proba` 가 클래스별 확률을 반환하므로 *얼마나 자신있게 예측했는가* 도 알 수 있습니다.

## 1D sin 회귀 — DecisionTreeRegressor vs KNeighborsRegressor

**목표**: 같은 1D sin 데이터에 두 회귀기를 fit해 출력이 실수임을 확인하고, tree와 k-NN의 곡선 모양 차이를 시각화합니다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
# 1) 1D sin 데이터 생성
rng = np.random.RandomState(0)
X = np.sort(5 * rng.rand(80, 1), axis=0)         # (80, 1)
y = np.sin(X).ravel() + rng.randn(80) * 0.1      # (80,) noise 포함
print("X.shape =", X.shape, "  y.shape =", y.shape, "  y.dtype =", y.dtype)

In [ ]:
# 시각화 — raw 데이터 분포 (fit 전에 봄)
fig, ax = plt.subplots(figsize=(6, 3))
ax.scatter(X, y, color="black", s=20)
ax.set_title("raw 1D sin + noise"); ax.set_xlabel("X"); ax.set_ylabel("y")
plt.tight_layout(); plt.show()

In [ ]:
# 2) 두 가족의 회귀기 fit
tree_shallow = DecisionTreeRegressor(max_depth=2).fit(X, y)
tree_deep    = DecisionTreeRegressor(max_depth=5).fit(X, y)
knn_uniform  = KNeighborsRegressor(n_neighbors=5, weights="uniform").fit(X, y)

In [ ]:
# 3) X_grid에 predict — 출력이 실수임을 dtype으로 확인
X_grid = np.linspace(0, 5, 500).reshape(-1, 1)
y_tree_shallow = tree_shallow.predict(X_grid)
y_tree_deep    = tree_deep.predict(X_grid)
y_knn          = knn_uniform.predict(X_grid)
print("y_tree_shallow.dtype =", y_tree_shallow.dtype, "  (회귀기 출력은 실수)")
print("y_tree_shallow[:3]   =", y_tree_shallow[:3].round(3))

In [ ]:
# 4) 세 곡선 비교
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X, y, color="black", s=20, label="data")
ax.plot(X_grid, y_tree_shallow, label="Tree depth=2", linestyle="--")
ax.plot(X_grid, y_tree_deep,    label="Tree depth=5", linestyle="-.")
ax.plot(X_grid, y_knn,          label="k-NN k=5",     linestyle=":")
ax.legend(); ax.set_title("1D sin 회귀 — tree vs k-NN")
plt.tight_layout(); plt.show()

scatter plot에서 raw 80개 점이 sin 곡선 모양의 분포를 보이고, 두 번째 plot에서 세 곡선이 함께 표시됩니다 — tree depth=2는 4단의 거친 plateau, tree depth=5는 noise까지 따라가는 흔들리는 곡선, k-NN k=5는 가장 부드러운 곡선.

**해석**: `predict.dtype` 가 `float64` 라는 한 줄이 *분류기와의 차이* 를 가장 분명히 보여 줍니다. tree depth=5는 *학습 데이터의 noise까지 흡수* 한 overfit의 첫 시각화이며 (§7 AP9의 plateau도 양 끝에서 관찰 가능), k-NN은 *k 이웃 평균* 이라 가장 부드럽습니다. 마지막 단계는 분류기의 결정 경계를 4-panel로 시각화합니다.


# iris 결정 경계 4-panel — DecisionBoundaryDisplay

**목표**: iris의 첫 두 feature 평면에 4개 estimator 가족(LogReg / SVC / k-NN / Tree)의 결정 경계를 동시에 그려 *모양 차이* 를 한 화면에서 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
iris = load_iris()
X = iris.data[:, :2]    # (150, 2): 꽃받침 길이/너비
y = iris.target         # (150,) ∈ {0,1,2}
print("X.shape =", X.shape, "  y unique =", np.unique(y))

In [ ]:
# 시각화 — raw 2D 평면 (fit 전에 봄)
fig, ax = plt.subplots(figsize=(5, 4))
for cls in np.unique(y):
    mask = y == cls
    ax.scatter(X[mask, 0], X[mask, 1], label=iris.target_names[cls])
ax.set_xlabel("sepal length"); ax.set_ylabel("sepal width")
ax.legend(); ax.set_title("iris 첫 두 feature — raw 분포")
plt.tight_layout(); plt.show()

In [ ]:
# 4 estimator 정의 + 결정 경계 4-panel
estimators_2d = [
    ("LogisticRegression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
    ("SVC (rbf)",          make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0))),
    ("KNeighbors (k=5)",   make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))),
    ("DecisionTree d=4",   DecisionTreeClassifier(max_depth=4, random_state=0)),
]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, est) in zip(axes.ravel(), estimators_2d):
    est.fit(X, y)
    DecisionBoundaryDisplay.from_estimator(
        est, X, response_method="predict", ax=ax, eps=0.5,
    )
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", s=20)
    ax.set_title(f"{name}  (acc={est.score(X, y):.3f})")
plt.tight_layout(); plt.show()


먼저 raw scatter plot이 표시되어 3개 클래스가 부분적으로 겹친 분포를 보이고, 이어서 4-panel figure가 표시됩니다 — LogReg는 직선 분리, SVC는 부드러운 곡선, k-NN은 들쭉날쭉 voronoi 모양, Tree는 축에 평행한 네모 영역. 각 panel 제목에 정확도가 0.78~0.85 범위로 표시됩니다.